# OLAF isolated smoke — DocRED + FinCausal + EventStoryLine

This is the **cheap first test**, not the final benchmark.

It runs one positive-gold document from each dataset with a two-LLM-call OLAF pipeline using
OpenRouter `openai/gpt-oss-20b` at minimal reasoning effort.

Gold annotations are used only to *select* a positive document and are stripped before OLAF execution.

### GPT-OSS format compatibility fix
OLAF's native LLM concept/relation components parse the model response with `ast.literal_eval()` and require a bare `list[list[str]]`. GPT-OSS may wrap the same answer in Markdown/explanatory text. The OpenRouter adapter now enforces and normalizes **format only**; it does not add benchmark or ontology knowledge.

### Candidate-vocabulary guard (IndexError fix)
GPT-OSS sometimes returns a syntactically valid group containing a renamed or invented term. OLAF silently converts such a group to an empty set and then `cts_to_concept()` crashes at `candidates[0]`. The adapter now keeps only terms from OLAF's own `Words :` candidate list and removes empty groups. This is an OLAF compatibility guard only; no benchmark labels or gold are exposed.


In [1]:
from pathlib import Path
import os, sys, json
from pprint import pprint

HERE = Path.cwd().resolve()

# Works whether Jupyter starts from the notebook folder or the olaf_baseline root.
BASE = HERE
while BASE.name != "olaf_baseline" and BASE.parent != BASE:
    BASE = BASE.parent
if BASE.name != "olaf_baseline":
    candidate = HERE.parent
    if candidate.name == "olaf_baseline":
        BASE = candidate
    else:
        raise RuntimeError("Could not locate olaf_baseline folder.")

SRC = BASE / "src"
VENDOR = BASE / "vendor" / "olaf"
for p in [SRC, VENDOR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from dotenv import load_dotenv
load_dotenv(BASE / ".env")

# ---- spaCy model preflight (same isolated Jupyter kernel) ----
import importlib
import subprocess
import spacy

SPACY_MODEL = "en_core_web_sm"

print("Kernel Python:", sys.executable)
if "olaf_baseline" not in str(sys.executable).lower() or ".venv" not in str(sys.executable).lower():
    raise RuntimeError(
        "Wrong Jupyter kernel. Select 'OLAF Baseline (.venv)' before continuing.\n"
        f"Current Python: {sys.executable}"
    )

def ensure_spacy_model(model_name: str) -> None:
    try:
        spacy.load(model_name)
        print(f"spaCy model ready: {model_name}")
        return
    except OSError as exc:
        if "[E050]" not in str(exc):
            raise

    print(f"spaCy model {model_name!r} is missing.")
    print("Installing it into THIS notebook kernel's .venv...")
    subprocess.check_call(
        [sys.executable, "-m", "spacy", "download", model_name]
    )
    importlib.invalidate_caches()

    # Verify immediately, before any paid OpenRouter calls.
    spacy.load(model_name)
    print(f"spaCy model installed and verified: {model_name}")

ensure_spacy_model(SPACY_MODEL)
# -------------------------------------------------------------


from dataset_io import (
    discover_ragtree_preprocessed, locate_dataset, read_jsonl,
    choose_positive_row, strip_gold, positive_gold_relation_count,
)
from olaf_lite_pipeline import run_olaf_lite_document

print("BASE:", BASE)
print("Python:", sys.executable)
print("Model:", os.getenv("OLAF_OPENROUTER_MODEL", "openai/gpt-oss-20b"))


Kernel Python: c:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\olaf_baseline\.venv\Scripts\python.exe
spaCy model ready: en_core_web_sm


c:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\olaf_baseline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BASE: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\olaf_baseline
Python: c:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\olaf_baseline\.venv\Scripts\python.exe
Model: openai/gpt-oss-20b


In [2]:
# Dataset discovery and zero-cost selection.
preprocessed = discover_ragtree_preprocessed(BASE)

paths = {
    k: locate_dataset(preprocessed, k)
    for k in ["docred", "fincausal", "eventstoryline"]
}

selected = {}
for k, p in paths.items():
    rows = read_jsonl(p)
    row = choose_positive_row(rows)
    selected[k] = row
    print(
        f"{k:15s} | file={p.name} | rows={len(rows)} | "
        f"id={row.get('document_id')} | gold_relations={positive_gold_relation_count(row)}"
    )

print("\nNo API calls made yet.")


docred          | file=docred_causal.jsonl | rows=106924 | id=DocRED - e37288ca6012859f | gold_relations=7
fincausal       | file=fincausal.jsonl | rows=967 | id=FinCausal - a39155e8dec741b9 | gold_relations=1
eventstoryline  | file=eventstoryline.jsonl | rows=443 | id=EventStoryLine - 1_10ecbplus | gold_relations=20

No API calls made yet.


In [3]:
# ZERO-COST GPT-OSS -> OLAF compatibility self-test.
from openrouter_generator import normalize_olaf_grouping_output

allowed = ["Skai TV", "Piraeus", "television network"]

cases = [
    ('[["Skai TV"], ["Piraeus"]]', "[['Skai TV'], ['Piraeus']]"),
    ('```json\n[["Skai TV"], ["Piraeus"]]\n```', "[['Skai TV'], ['Piraeus']]"),
    ('Here is the result:\n[["Skai TV"], ["Piraeus"]]', "[['Skai TV'], ['Piraeus']]"),
    ('{"groups": [["Skai TV"], ["Piraeus"]]}', "[['Skai TV'], ['Piraeus']]"),
    # Exact bug from the failed run: an invented label would create an
    # empty candidate set inside OLAF and cts_to_concept() would index [0].
    ('[["Skai TV"], ["invented concept"]]', "[['Skai TV']]"),
    # harmless case/whitespace variation may map uniquely back to exact prompt string
    ('[["  skai   tv  "], ["PIRAEUS"]]', "[['Skai TV'], ['Piraeus']]"),
    # if every label is invalid, return [] rather than [[]] so OLAF does not crash
    ('[["invented one"], ["invented two"]]', "[]"),
]

for raw, expected in cases:
    normalized, dropped = normalize_olaf_grouping_output(
        raw, allowed_labels=allowed
    )
    assert normalized == expected, (raw, normalized, expected)

print("GPT-OSS -> OLAF syntax + candidate-vocabulary guard: PASSED")
print("Empty candidate groups are removed before OLAF sees them.")
print("No benchmark labels/gold are used. No API calls made.")


GPT-OSS -> OLAF syntax + candidate-vocabulary guard: PASSED
Empty candidate groups are removed before OLAF sees them.
No benchmark labels/gold are used. No API calls made.


## Paid smoke

Expected cost profile: roughly **2 LLM calls per document**, 6 calls total.

This is intentionally much lighter than NeoOLAF.


In [4]:
if not os.getenv("OPENROUTER_API_KEY", "").strip():
    raise RuntimeError("OPENROUTER_API_KEY is missing.")

RUN_DIR = BASE / "runs" / "smoke_one_each"
RUN_DIR.mkdir(parents=True, exist_ok=True)

results = {}

for dataset_key in ["docred", "fincausal", "eventstoryline"]:
    gold_row = selected[dataset_key]
    clean = strip_gold(gold_row)

    forbidden = {"entities", "relations", "pred_relations", "ontology_links"} & set(clean)
    assert not forbidden, forbidden

    print(f"\n=== {dataset_key.upper()} ===")
    result = run_olaf_lite_document(
        clean["text"],
        model_name=os.getenv("OLAF_OPENROUTER_MODEL", "openai/gpt-oss-20b"),
        reasoning_effort=os.getenv("OLAF_REASONING_EFFORT", "minimal"),
    )

    payload = {
        "dataset": dataset_key,
        "document_id": gold_row.get("document_id"),
        "title": gold_row.get("title"),
        "gold_relation_count_posthoc_only": positive_gold_relation_count(gold_row),
        "elapsed_seconds": result.elapsed_seconds,
        "concepts": result.concepts,
        "relations": result.relations,
    }
    results[dataset_key] = payload

    out = RUN_DIR / f"{dataset_key}.json"
    out.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

    print("elapsed_seconds:", round(result.elapsed_seconds, 2))
    print("concepts:", len(result.concepts))
    print("relations:", len(result.relations))
    print("first relations:")
    pprint(result.relations[:10])

print("\nSaved:", RUN_DIR)



=== DOCRED ===


[2026-08-25 00:11:40,284] [WARNING] [pos_term_extraction] [__init__] [No preprocessing function provided for spans. Using the default one.]
[2026-08-25 00:11:40,286] [WARNING] [pos_term_extraction] [_check_parameters] [POS term extraction token sequence attribute not set by the user.
               By default the system will use the entire content of the document.]
[2026-08-25 00:11:40,286] [WARNING] [pos_term_extraction] [__init__] [No preprocessing function provided for spans. Using the default one.]
[2026-08-25 00:11:40,287] [WARNING] [pos_term_extraction] [_check_parameters] [POS term extraction token sequence attribute not set by the user.
               By default the system will use the entire content of the document.]


elapsed_seconds: 12.97
concepts: 10
relations: 7
first relations:
[{'linguistic_realisations': ['introducing'],
  'occurrences': ['introducing'],
  'predicate': 'introducing',
  'source': None,
  'target': None},
 {'linguistic_realisations': ['is'],
  'occurrences': ['is'],
  'predicate': 'is',
  'source': None,
  'target': None},
 {'linguistic_realisations': ['opted'],
  'occurrences': ['opted'],
  'predicate': 'opted',
  'source': None,
  'target': None},
 {'linguistic_realisations': ['relaunched'],
  'occurrences': ['relaunched'],
  'predicate': 'relaunched',
  'source': None,
  'target': None},
 {'linguistic_realisations': ['spread'],
  'occurrences': ['spread'],
  'predicate': 'spread',
  'source': None,
  'target': None},
 {'linguistic_realisations': ['switched'],
  'occurrences': ['switched'],
  'predicate': 'switched',
  'source': None,
  'target': None},
 {'linguistic_realisations': ['was'],
  'occurrences': ['was'],
  'predicate': 'was',
  'source': None,
  'target': None}]



[2026-08-25 00:11:54,208] [WARNING] [pos_term_extraction] [__init__] [No preprocessing function provided for spans. Using the default one.]
[2026-08-25 00:11:54,210] [WARNING] [pos_term_extraction] [_check_parameters] [POS term extraction token sequence attribute not set by the user.
               By default the system will use the entire content of the document.]
[2026-08-25 00:11:54,211] [WARNING] [pos_term_extraction] [__init__] [No preprocessing function provided for spans. Using the default one.]
[2026-08-25 00:11:54,212] [WARNING] [pos_term_extraction] [_check_parameters] [POS term extraction token sequence attribute not set by the user.
               By default the system will use the entire content of the document.]


elapsed_seconds: 5.17
concepts: 7
relations: 14
first relations:
[{'linguistic_realisations': ['adjusted'],
  'occurrences': ['adjusted'],
  'predicate': 'adjusted',
  'source': 'combined',
  'target': 'agi'},
 {'linguistic_realisations': ['aged', 'is', 'remaining'],
  'occurrences': ['aged', 'is', 'remaining'],
  'predicate': 'aged',
  'source': None,
  'target': None},
 {'linguistic_realisations': ['came'],
  'occurrences': ['came'],
  'predicate': 'came',
  'source': 'aged',
  'target': 'aged'},
 {'linguistic_realisations': ['drew'],
  'occurrences': ['drew'],
  'predicate': 'drew',
  'source': 'state',
  'target': 'combined'},
 {'linguistic_realisations': ['from'],
  'occurrences': ['from'],
  'predicate': 'from',
  'source': 'aged',
  'target': 'aged'},
 {'linguistic_realisations': ['from'],
  'occurrences': ['from'],
  'predicate': 'from',
  'source': 'came',
  'target': 'aged'},
 {'linguistic_realisations': ['combined', 'for', 'in', 'of', 'to'],
  'occurrences': ['combined', 'fo

[2026-08-25 00:12:00,397] [WARNING] [pos_term_extraction] [__init__] [No preprocessing function provided for spans. Using the default one.]
[2026-08-25 00:12:00,399] [WARNING] [pos_term_extraction] [_check_parameters] [POS term extraction token sequence attribute not set by the user.
               By default the system will use the entire content of the document.]
[2026-08-25 00:12:00,400] [WARNING] [pos_term_extraction] [__init__] [No preprocessing function provided for spans. Using the default one.]
[2026-08-25 00:12:00,400] [WARNING] [pos_term_extraction] [_check_parameters] [POS term extraction token sequence attribute not set by the user.
               By default the system will use the entire content of the document.]


elapsed_seconds: 18.91
concepts: 9
relations: 12
first relations:
[{'linguistic_realisations': ['begin'],
  'occurrences': ['begin'],
  'predicate': 'begin',
  'source': None,
  'target': None},
 {'linguistic_realisations': ['checked'],
  'occurrences': ['checked'],
  'predicate': 'checked',
  'source': None,
  'target': None},
 {'linguistic_realisations': ['checks'],
  'occurrences': ['checks'],
  'predicate': 'checks',
  'source': None,
  'target': None},
 {'linguistic_realisations': ['driving'],
  'occurrences': ['driving'],
  'predicate': 'driving',
  'source': None,
  'target': None},
 {'linguistic_realisations': ['ended'],
  'occurrences': ['ended'],
  'predicate': 'ended',
  'source': None,
  'target': None},
 {'linguistic_realisations': ['facing'],
  'occurrences': ['facing'],
  'predicate': 'facing',
  'source': 'entering',
  'target': 'arrest'},
 {'linguistic_realisations': ['facing'],
  'occurrences': ['facing'],
  'predicate': 'facing',
  'source': 'entering',
  'target': '

## Inspection summary

Send me the executed notebook or the three JSON files from `runs/smoke_one_each`.

The next step is to freeze the **small benchmark projection layer**:
map OLAF source/target concept occurrences to dataset gold endpoint spans, and map OLAF lexical relation labels
to the allowed relation IDs. Once that adapter is verified on a few documents, we can run the full three datasets
without giving OLAF the much heavier NeoOLAF orchestration.


In [5]:
import pandas as pd

summary = pd.DataFrame([
    {
        "dataset": k,
        "document_id": v["document_id"],
        "elapsed_seconds": v["elapsed_seconds"],
        "concept_count": len(v["concepts"]),
        "relation_count": len(v["relations"]),
        "gold_relation_count_posthoc_only": v["gold_relation_count_posthoc_only"],
    }
    for k, v in results.items()
])
display(summary)


,dataset,document_id,elapsed_seconds,concept_count,relation_count,gold_relation_count_posthoc_only
0,docred,DocRED - e37288ca6012859f,12.966543,10,7,7
1,fincausal,FinCausal - a39155e8dec741b9,5.169509,7,14,1
2,eventstoryline,EventStoryLine - 1_10ecbplus,18.905587,9,12,20
